In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

docs = pd.read_csv("document.csv")
queries = pd.read_csv("query.csv")
print(docs.shape, queries.shape)

(20000, 2) (200, 2)


In [2]:
def clean(s):
    s = str(s).replace("ي", "ی").replace("ك", "ک").replace("‌", " ")
    s = s.translate(str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789"))
    s = re.sub(r"[ً-ٟـ]", "", s)
    return s.lower()


def words(s):
    return re.findall(r"\w+", clean(s))


doc_words = [words(t) for t in docs["text"]]
heads = [" ".join(w[:25]) for w in doc_words]
bodies = [" ".join(w[:600]) for w in doc_words]

In [3]:
head_vec = TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), sublinear_tf=True, min_df=1)
body_vec = TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_df=0.5)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), sublinear_tf=True)
head_mat = head_vec.fit_transform(heads)
char_mat = char_vec.fit_transform([" ".join(w[:8]) for w in doc_words])
body_mat = body_vec.fit_transform(bodies)

In [4]:
starts = {}
for i, w in enumerate(doc_words):
    for k in (2, 3, 4, 6, 10):
        starts.setdefault(tuple(w[:k]), []).append(i)


def split_title(q):
    m = re.match(r"^(.*?)\s*\((.*)\)\s*$", q)
    if m and m.group(1).strip():
        return m.group(1), m.group(2)
    return q, ""


def prefix_bonus(q):
    bonus = np.zeros(len(docs))
    core, extra = split_title(q)
    cw = words(core)
    ew = words(extra)
    qw = words(q)
    if len(qw) >= 10:
        for k in (10, 6):
            for i in starts.get(tuple(qw[:k]), []):
                bonus[i] += 30
            if bonus.max() > 0:
                break
    if cw and len(cw) <= 10 and len(qw) < 10:
        for i, w in enumerate(doc_words):
            if w[:len(cw)] == cw:
                bonus[i] += 8
                if ew and all(x in w[:60] for x in ew):
                    bonus[i] += 3
    return bonus

In [5]:
rows = []
for qi, q in zip(queries["query_index"], queries["query"]):
    core, extra = split_title(q)
    qh = head_vec.transform([" ".join(words(q))])
    qb = body_vec.transform([" ".join(words(q))])
    qc = char_vec.transform([" ".join(words(core))])
    score = (3.0 * (head_mat @ qh.T).toarray().ravel()
             + (body_mat @ qb.T).toarray().ravel()
             + 2.0 * (char_mat @ qc.T).toarray().ravel())
    score = score + prefix_bonus(q)
    top = np.argsort(-score, kind="stable")[:5]
    rows.append((qi, "|".join(docs["index"].iloc[top])))

result = pd.DataFrame(rows, columns=["query_index", "document_indexes"])
result.to_csv("result.csv", index=False)
print(result.head())

   query_index                                   document_indexes
0            1  2d91e5e2a2528126|78cbf19b57a18c76|fba7ea1d02ae...
1            2  b65914ac878201df|c71e16a36f52a8d1|fdd046e9fc2b...
2            3  b13dac6f69200623|fb7ec540ade449f6|07598073f4c9...
3            4  1bca71e12ab6925d|69f853bd114f8f96|2568364cb260...
4            5  9903150227debc26|197ec3921dc3d6de|3738236e678e...
